<a href="https://colab.research.google.com/github/kuteesatendojeremiah/decodelabs_DS_intern_work/blob/main/project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS Intern Work — Project 2: Fraud Detection Pipeline (Supervised Learning)

Goal: train and tune a classifier to catch fraudulent transactions in a highly imbalanced dataset (~99.83% legitimate / ~0.17% fraud), using the [Kaggle Credit Card Fraud Detection dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).

**Zero-Leakage Protocol:**
- Stratified train/test split first, before anything else touches the data.
- SMOTE and scaling live *inside* the pipeline, so cross-validation only ever resamples/scales the training fold, never the validation fold, never the test set.
- Test set stays untouched (real-world imbalance) until final evaluation.
- No accuracy metric. Precision, Recall, ROC-AUC, and a Confusion Matrix on the held-out test set only.

In [1]:
# Install dependencies (imbalanced-learn for SMOTE + imblearn.pipeline, kagglehub for dataset download)
!pip install imbalanced-learn kagglehub -q

## 1. Load the data

Downloads the dataset straight from Kaggle via `kagglehub`. On first run in Colab this will prompt you to authenticate (either `kagglehub.login()` interactively, or upload a `kaggle.json` API token beforehand, see the Kaggle API docs).

In [2]:
import os
import kagglehub

dataset_path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print("Dataset downloaded to:", dataset_path)
print(os.listdir(dataset_path))

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Dataset downloaded to: /kaggle/input/creditcardfraud
['creditcard.csv']


In [3]:
import pandas as pd

csv_path = os.path.join(dataset_path, "creditcard.csv")
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
df.head()

Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## EDA: missing values, dtypes, duplicates

Quick data-integrity pass before anything else touches the data. `V1`-`V28` are already PCA components (anonymized), so there's no raw categorical cleanup to do here, but we still check for nulls, dtype issues, and duplicate rows rather than assume they're absent.

In [4]:
df.info()
print()
print("Missing values per column:")
print(df.isnull().sum().sum(), "total missing values")
print(df.isnull().sum()[df.isnull().sum() > 0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [5]:
df.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.168375e-15,3.416908e-16,-1.379537e-15,2.074095e-15,9.604066e-16,1.487313e-15,-5.556467e-16,1.213481e-16,-2.406331e-15,...,1.654067e-16,-3.568593e-16,2.578648e-16,4.473266e-15,5.340915e-16,1.683437e-15,-3.660091e-16,-1.227390e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


In [6]:
# Duplicate rows: this dataset is known to contain exact duplicates
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes} ({n_dupes / len(df):.4%} of data)")

Duplicate rows: 1081 (0.3796% of data)


Duplicates matter here specifically because of the zero-leakage protocol: if a duplicate row exists, the split could place one copy in `X_train` and its exact twin in `X_test`, which is leakage (the model would effectively be evaluated on a row it already trained on). Dropping duplicates now, before the split, is the correct place to do it.

In [7]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)

Shape after dropping duplicates: (283726, 31)


In [8]:
# Confirm the class imbalance (after deduping)
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100

print(class_counts)
print()
print(class_pct.round(4))

Class
0    283253
1       473
Name: count, dtype: int64

Class
0    99.8333
1     0.1667
Name: proportion, dtype: float64


## 2. Stratified train/test split, before any processing

This is the only split in the entire notebook. Everything downstream (scaling, SMOTE, hyperparameter search) happens only on `X_train`/`y_train`. `X_test`/`y_test` stay untouched and keep the real-world 99.83/0.17 imbalance until final evaluation.

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape, "Fraud rate:", y_train.mean())
print("Test shape: ", X_test.shape, "Fraud rate:", y_test.mean())

Train shape: (226980, 30) Fraud rate: 0.0016653449643140364
Test shape:  (56746, 30) Fraud rate: 0.0016741268107003137


## 3. Build the pipelines

Both pipelines use `imblearn.pipeline.Pipeline` (not `sklearn.pipeline.Pipeline`) so that SMOTE is treated as a proper pipeline step: during cross-validation, imblearn only fits/applies SMOTE on the training fold of each split and leaves the validation fold at its natural imbalance.

- **Logistic Regression:** `StandardScaler` then `SMOTE` then `LogisticRegression` (LR is scale-sensitive, so `Amount`/`Time` need scaling before SMOTE interpolates neighbors).
- **Random Forest:** `SMOTE` then `RandomForestClassifier` (tree splits are scale-invariant, so no scaler).

In [10]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

lr_pipeline = ImbPipeline(steps=[
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

rf_pipeline = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

## 4. Hyperparameter tuning with GridSearchCV

Tuning is holistic: `smote__k_neighbors` is searched jointly with the classifier hyperparameters, inside the same cross-validated pipeline, so no combination of resampling and model settings is chosen using leaked validation information.

`StratifiedKFold` keeps each CV fold's class ratio representative of the training set. Scoring tracks Precision, Recall, and ROC-AUC together; refit selects the best model by ROC-AUC (a threshold-independent measure of separability), and precision/recall are still inspected per model below.

In [11]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["precision", "recall", "roc_auc"]

lr_param_grid = {
    "smote__k_neighbors": [3, 5, 7],
    "clf__C": [0.01, 0.1, 1, 10],
}

rf_param_grid = {
    "smote__k_neighbors": [3, 5, 7],
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [5, 10, None],
}

lr_search = GridSearchCV(
    lr_pipeline, lr_param_grid,
    scoring=scoring, refit="roc_auc",
    cv=cv, n_jobs=-1, verbose=1
)

rf_search = GridSearchCV(
    rf_pipeline, rf_param_grid,
    scoring=scoring, refit="roc_auc",
    cv=cv, n_jobs=-1, verbose=1
)

In [12]:
# Fit Logistic Regression grid search (SMOTE + scaling applied only within each training fold)
lr_search.fit(X_train, y_train)

print("Best LR params:", lr_search.best_params_)
print("Best LR CV ROC-AUC:", lr_search.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best LR params: {'clf__C': 0.01, 'smote__k_neighbors': 3}
Best LR CV ROC-AUC: 0.9816394949014267


In [ ]:
# Fit Random Forest grid search (SMOTE applied only within each training fold)
rf_search.fit(X_train, y_train)

print("Best RF params:", rf_search.best_params_)
print("Best RF CV ROC-AUC:", rf_search.best_score_)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


## 5. Final evaluation on the untouched test set

`X_test`/`y_test` have never been seen by SMOTE, the scaler, or GridSearchCV, so they still reflect the real 99.83/0.17 imbalance. No accuracy metric: Precision, Recall, ROC-AUC, and a Confusion Matrix only.

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

def evaluate(name, fitted_search, X_test, y_test):
    best_model = fitted_search.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    print(f"=== {name} ===")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate", "Fraud"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"{name} - Confusion Matrix (test set)")
    plt.show()

    return {"model": name, "precision": precision, "recall": recall, "roc_auc": roc_auc}

lr_results = evaluate("Logistic Regression", lr_search, X_test, y_test)
rf_results = evaluate("Random Forest", rf_search, X_test, y_test)

In [ ]:
# Side-by-side comparison
results_df = pd.DataFrame([lr_results, rf_results]).set_index("model")
results_df